# PE Target Screener — Exploratory Analysis

This notebook walks through the screening logic step by step.
Use it to understand the data, validate ratios, and explore scoring decisions.

**Run `python main.py` first to generate the processed dataset.**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yaml
import sys
sys.path.append('..')

pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 120)

with open('../config.yaml') as f:
    cfg = yaml.safe_load(f)

print('Config loaded')

## 1. Load Data

In [ ]:
# Load processed data (after running main.py)
try:
    df = pd.read_csv('../data/processed/companies_scored.csv')
    print(f'Loaded {len(df)} companies')
except FileNotFoundError:
    print('Run python main.py first to generate data')
    df = None

if df is not None:
    display(df.head())

## 2. Universe Overview

In [ ]:
if df is not None:
    print('Sector breakdown:')
    print(df['sector'].value_counts())
    print()
    print('Debt capacity distribution:')
    print(df['debt_capacity'].value_counts())
    print()
    print('Companies with red flags:', (df['red_flags'] != '').sum())

## 3. Ratio Distributions

In [ ]:
if df is not None:
    ratios = ['ebitda_margin', 'fcf_conversion', 'net_debt_to_ebitda', 'interest_coverage', 'ev_to_ebitda', 'roic']
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    axes = axes.flatten()
    
    for i, col in enumerate(ratios):
        if col in df.columns:
            clean_data = df[col].dropna()
            # Clip outliers for display
            p5, p95 = clean_data.quantile([0.05, 0.95])
            clean_data = clean_data.clip(p5, p95)
            axes[i].hist(clean_data, bins=20, color='steelblue', edgecolor='white', alpha=0.8)
            axes[i].set_title(col.replace('_', ' ').title(), fontsize=11)
            axes[i].axvline(clean_data.median(), color='red', linestyle='--', label=f'Median: {clean_data.median():.2f}')
            axes[i].legend(fontsize=8)
    
    plt.tight_layout()
    plt.suptitle('Distribution of Key PE Metrics', fontsize=14, y=1.02)
    plt.savefig('../outputs/ratio_distributions.png', dpi=150, bbox_inches='tight')
    plt.show()

## 4. Top 20 Targets

In [ ]:
if df is not None and 'rank' in df.columns:
    top20 = df[df['rank'] <= 20][[
        'rank', 'company', 'sector', 'ebitda_margin', 'fcf_conversion',
        'net_debt_to_ebitda', 'interest_coverage', 'ev_to_ebitda',
        'debt_capacity', 'pe_score'
    ]].copy()
    
    for col in ['ebitda_margin', 'fcf_conversion']:
        if col in top20.columns:
            top20[col] = top20[col].apply(lambda x: f'{x:.1%}' if pd.notna(x) else '—')
    for col in ['net_debt_to_ebitda', 'interest_coverage', 'ev_to_ebitda']:
        if col in top20.columns:
            top20[col] = top20[col].apply(lambda x: f'{x:.1f}x' if pd.notna(x) else '—')
    
    display(top20.set_index('rank'))

## 5. Investment Memos — Top 5

In [ ]:
if df is not None and 'investment_memo' in df.columns:
    for _, row in df[df['rank'] <= 5].iterrows():
        print(row.get('investment_memo', ''))
        print()

## 6. Score vs Valuation Scatter

In [ ]:
if df is not None:
    plot_df = df.dropna(subset=['pe_score', 'ev_to_ebitda', 'ebitda_margin'])
    plot_df = plot_df[plot_df['ev_to_ebitda'] < 30]  # Remove extreme outliers
    
    colors = {'High': '#2ecc71', 'Medium': '#f39c12', 'Low': '#e74c3c'}
    
    fig, ax = plt.subplots(figsize=(12, 7))
    for dc, group in plot_df.groupby('debt_capacity'):
        ax.scatter(
            group['ev_to_ebitda'], group['pe_score'],
            s=group['ebitda_margin'].clip(0.05, 0.35) * 800,
            c=colors.get(dc, 'gray'), label=dc, alpha=0.7, edgecolors='white'
        )
        for _, row in group.nlargest(3, 'pe_score').iterrows():
            ax.annotate(row.get('ticker', ''), (row['ev_to_ebitda'], row['pe_score']),
                        fontsize=8, ha='center', va='bottom')
    
    ax.set_xlabel('EV / EBITDA (lower = cheaper entry)', fontsize=11)
    ax.set_ylabel('PE Score (higher = more attractive)', fontsize=11)
    ax.set_title('PE Score vs Entry Valuation\n(bubble size = EBITDA margin)', fontsize=13)
    ax.legend(title='Debt Capacity')
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig('../outputs/score_vs_valuation.png', dpi=150, bbox_inches='tight')
    plt.show()